# Stochastic Gradient Descent

This notebook explores **stochastic gradient descent (SGD)**, a fundamental optimization algorithm used to minimize loss functions in machine learning. Unlike batch gradient descent which uses the entire dataset, SGD updates parameters using one sample at a time, making it faster for large datasets but introducing noise into the optimization process.

## Learning Objectives
By the end of this notebook, you will be able to:
1. **Explain** why SGD uses a single sample per update and how this makes each iteration $O(p)$ instead of $O(np)$.
2. **Implement** the SGD update $\theta \leftarrow \theta - \alpha \nabla \ell_i(\theta)$ and shuffle training data between epochs.
3. **Describe** why SGD gradient estimates are unbiased but high-variance, and connect this to the oscillating loss curve.
4. **Apply** a learning rate schedule (e.g., $\alpha_t = \alpha_0 / (1 + t)$) and explain why constant $\alpha$ prevents exact convergence.
5. **Compare** SGD vs BGD vs Mini-Batch GD across convergence speed, stability, and memory requirements.
6. **Recognize** when SGD's noise can help escape local minima in non-convex objectives.

> **Prerequisite**: Batch gradient descent ([ml_001_02](ml_001_02_batch_gradient_descent.ipynb)).

## Important Note on Mathematical Notation

Throughout this notebook, whenever we use a mathematical concept or formula (such as derivatives, gradients, or optimization updates), we first present the **general foundational principle** before applying it to our specific problem. This approach ensures clarity by:

1. Introducing the general concept and formula
2. Explaining what each component means
3. Showing how we apply it to stochastic gradient descent

This makes it easier to understand not just *how* SGD works, but *why* it works and where it comes from.

## 1. Introduction to Stochastic Gradient Descent

### 1.1 Notation Review: Three Ways to Write the Loss

**Scalar form** (per-sample loss, squared error):
$$\ell_i(\theta) = \frac{1}{2}(y_i - \theta^T x_i)^2$$
where $x_i \in \mathbb{R}^p$ and $y_i \in \mathbb{R}$ are the features and label for sample $i$.

**Sum form** (full-dataset loss):
$$J(\theta) = \frac{1}{n}\sum_{i=1}^{n} \ell_i(\theta)$$

**Matrix form** (compact):
$$J(\theta) = \frac{1}{n}\|y - X\theta\|^2, \quad X \in \mathbb{R}^{n \times p},\; y \in \mathbb{R}^n$$

### 1.2 From Batch to Stochastic

**Batch Gradient Descent (BGD)** uses all $n$ samples per update:
$$\theta := \theta - \alpha \cdot \frac{1}{n}\sum_{i=1}^{n} \nabla_\theta \ell_i(\theta)$$
Cost per step: $O(np)$. For $n = 10^6, p = 1000$: $10^9$ FLOPs before taking even one step.

**Stochastic Gradient Descent (SGD)** uses one randomly chosen sample $i$:
$$\theta := \theta - \alpha \cdot \nabla_\theta \ell_i(\theta)$$
Cost per step: $O(p)$. SGD takes $n$ parameter steps per epoch vs BGD's 1.

### 1.3 BGD vs SGD Comparison

| Aspect | BGD | SGD |
|--------|-----|-----|
| **Gradient** | Exact $\nabla J(\theta)$ | Unbiased estimate $\nabla \ell_i(\theta)$ |
| **Cost per update** | $O(np)$ | $O(p)$ |
| **Updates per epoch** | 1 | $n$ |
| **Loss curve** | Smooth, monotone | Oscillating, noisy |
| **Convergence** | To exact minimum (convex) | To neighborhood of minimum |
| **Memory** | Full dataset | One sample |

### 1.4 Why the SGD Gradient is Unbiased

If sample index $i$ is drawn uniformly from $\{1, \ldots, n\}$:
$$\mathbb{E}_i\bigl[\nabla_\theta \ell_i(\theta)\bigr] = \frac{1}{n}\sum_{i=1}^{n} \nabla_\theta \ell_i(\theta) = \nabla J(\theta)$$

SGD points in the right direction on average — noise averages out over many steps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Generate synthetic linear regression dataset
n, p = 500, 5
X_raw = np.random.randn(n, p)
# prepend intercept column
X = np.hstack([np.ones((n, 1)), X_raw])
theta_true = np.array([1.0, 2.0, -1.5, 0.5, 1.0, -0.8])
y = X @ theta_true + 0.3 * np.random.randn(n)


def mse(X, y, theta):
    residuals = y - X @ theta
    return np.mean(residuals ** 2)


def bgd(X, y, alpha=0.01, n_epochs=50):
    n_samples, n_features = X.shape
    theta = np.zeros(n_features)
    losses = []
    for _ in range(n_epochs):
        losses.append(mse(X, y, theta))
        # full-batch gradient: O(np) cost per step
        grad = -2 / n_samples * X.T @ (y - X @ theta)
        theta -= alpha * grad
    return theta, losses


def sgd(X, y, alpha=0.005, n_epochs=50):
    n_samples, n_features = X.shape
    theta = np.zeros(n_features)
    losses = []
    indices = np.arange(n_samples)
    for _ in range(n_epochs):
        losses.append(mse(X, y, theta))
        # shuffle each epoch to avoid cyclic bias
        np.random.shuffle(indices)
        for i in indices:
            xi, yi = X[i], y[i]
            # single-sample gradient: O(p) cost per step
            grad_i = -2 * xi * (yi - xi @ theta)
            theta -= alpha * grad_i
    return theta, losses


theta_bgd, losses_bgd = bgd(X, y, alpha=0.01, n_epochs=50)
theta_sgd, losses_sgd = sgd(X, y, alpha=0.005, n_epochs=50)
# optimal solution via Normal Equation for baseline
optimal = mse(X, y, np.linalg.solve(X.T @ X, X.T @ y))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(losses_bgd, "b-", lw=2.5, label="BGD")
axes[0].plot(losses_sgd, "r-", lw=1.5, alpha=0.85, label="SGD")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].set_title("BGD vs SGD: Loss per Epoch\n(BGD smooth, SGD oscillating)")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].semilogy([l - optimal + 1e-10 for l in losses_bgd], "b-", lw=2.5, label="BGD")
axes[1].semilogy([l - optimal + 1e-10 for l in losses_sgd], "r-", lw=1.5, alpha=0.85, label="SGD")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Excess loss J(θ) − J* (log scale)")
axes[1].set_title("Convergence Comparison")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Optimal loss (Normal Eq.): {optimal:.6f}")
print(f"BGD final loss:            {losses_bgd[-1]:.6f}")
print(f"SGD final loss:            {losses_sgd[-1]:.6f}")
print(f"\nTrue θ:  {theta_true}")
print(f"BGD θ̂:   {np.round(theta_bgd, 3)}")
print(f"SGD θ̂:   {np.round(theta_sgd, 3)}")

## 2. Learning Rate Schedules and Mini-Batch SGD

### 2.1 Why Constant Learning Rate Prevents Exact Convergence

With a constant $\alpha$, SGD never fully converges: the gradient noise $\nabla \ell_i(\theta) - \nabla J(\theta)$ keeps bouncing $\theta$ around the optimum. The residual error is approximately:
$$\mathbb{E}[J(\theta_T) - J(\theta^*)] \geq C \cdot \alpha \cdot \sigma^2_{\text{grad}}$$
where $\sigma^2_{\text{grad}}$ is the gradient variance. To guarantee convergence to the exact minimum, the learning rate must satisfy the **Robbins-Monro conditions**:
$$\sum_{t=1}^{\infty} \alpha_t = \infty \qquad \text{and} \qquad \sum_{t=1}^{\infty} \alpha_t^2 < \infty$$
Common schedules that satisfy these: $\alpha_t = \alpha_0/t$ or $\alpha_t = \alpha_0/\sqrt{t}$.

### 2.2 Mini-Batch SGD

Mini-batch SGD uses $b$ samples per update:
$$\theta := \theta - \frac{\alpha}{b} \sum_{i \in \mathcal{B}} \nabla_\theta \ell_i(\theta)$$

As batch size $b$ grows from 1 to $n$:

| Batch size | Gradient variance | GPU parallelism | Steps per epoch |
|-----------|-------------------|-----------------|-----------------|
| $b=1$ (SGD) | High ($\sim\sigma^2$) | Low | $n$ |
| $b=32$–$256$ | Low ($\sim\sigma^2/b$) | High | $n/b$ |
| $b=n$ (BGD) | Zero (exact) | Full (one kernel) | 1 |

In practice, $b \in \{32, 64, 128, 256\}$ balances variance reduction with hardware throughput.

In [ ]:
def sgd_with_schedule(X, y, alpha0=0.1, n_epochs=50, schedule="constant"):
    n_samples, n_features = X.shape
    theta = np.zeros(n_features)
    losses = []
    indices = np.arange(n_samples)
    total_iters = 0
    max_iters = n_epochs * n_samples
    for epoch in range(n_epochs):
        losses.append(mse(X, y, theta))
        np.random.shuffle(indices)
        for i in indices:
            if schedule == "constant":
                alpha = alpha0
            elif schedule == "sqrt":
                # 1/sqrt(t) decay satisfies Robbins-Monro conditions
                alpha = alpha0 / np.sqrt(total_iters + 1)
            elif schedule == "linear":
                # linear warmdown to alpha0/100
                alpha = alpha0 * (1 - total_iters / max_iters) + alpha0 / 100
            xi, yi = X[i], y[i]
            grad_i = -2 * xi * (yi - xi @ theta)
            theta -= alpha * grad_i
            total_iters += 1
    return theta, losses


def mini_batch_sgd(X, y, batch_size=32, alpha=0.01, n_epochs=50):
    n_samples, n_features = X.shape
    theta = np.zeros(n_features)
    losses = []
    indices = np.arange(n_samples)
    for _ in range(n_epochs):
        losses.append(mse(X, y, theta))
        np.random.shuffle(indices)
        for start in range(0, n_samples, batch_size):
            batch_idx = indices[start : start + batch_size]
            Xb, yb = X[batch_idx], y[batch_idx]
            # mini-batch gradient: average over b samples
            grad = -2 / len(batch_idx) * Xb.T @ (yb - Xb @ theta)
            theta -= alpha * grad
    return theta, losses


_, losses_const = sgd_with_schedule(X, y, alpha0=0.005, n_epochs=50, schedule="constant")
_, losses_sqrt = sgd_with_schedule(X, y, alpha0=0.05, n_epochs=50, schedule="sqrt")
_, losses_linear = sgd_with_schedule(X, y, alpha0=0.02, n_epochs=50, schedule="linear")

_, losses_b1 = mini_batch_sgd(X, y, batch_size=1, alpha=0.005, n_epochs=50)
_, losses_b16 = mini_batch_sgd(X, y, batch_size=16, alpha=0.008, n_epochs=50)
_, losses_b64 = mini_batch_sgd(X, y, batch_size=64, alpha=0.012, n_epochs=50)
_, losses_full = bgd(X, y, alpha=0.01, n_epochs=50)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(losses_const, "b-", lw=2, label="Constant α=0.005")
axes[0].plot(losses_sqrt, "r-", lw=2, label="1/√t decay")
axes[0].plot(losses_linear, "g-", lw=2, label="Linear warmdown")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].set_title("Learning Rate Schedules\n(SGD, batch size = 1)")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_ylim(bottom=0)

axes[1].plot(losses_b1, "r-", lw=1.5, alpha=0.8, label="batch=1 (SGD)")
axes[1].plot(losses_b16, color="orange", lw=2, label="batch=16")
axes[1].plot(losses_b64, "g-", lw=2, label="batch=64")
axes[1].plot(losses_full, "b-", lw=2.5, label="batch=n (BGD)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MSE Loss")
axes[1].set_title("Mini-Batch Size vs Convergence")
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_ylim(bottom=0)

plt.tight_layout()
plt.show()

print("Learning rate schedule comparison (final loss after 50 epochs):")
print(f"  Constant α:      {losses_const[-1]:.6f}")
print(f"  1/√t decay:      {losses_sqrt[-1]:.6f}")
print(f"  Linear warmdown: {losses_linear[-1]:.6f}")
print("\nMini-batch size comparison (final loss):")
print(f"  batch=1  (SGD): {losses_b1[-1]:.6f}")
print(f"  batch=16:       {losses_b16[-1]:.6f}")
print(f"  batch=64:       {losses_b64[-1]:.6f}")
print(f"  batch=n  (BGD): {losses_full[-1]:.6f}")

---
## Practice Exercises

**Conceptual**

1. SGD uses a single sample's gradient as an estimate of the true gradient. Show that $\mathbb{E}[\nabla \ell_i(\theta)] = \nabla J(\theta)$ — i.e., the SGD gradient estimate is unbiased.

2. Why does SGD with a constant learning rate $\alpha$ not converge to the exact minimum, even for a convex loss? What learning rate schedule fixes this?

3. Explain the "epoch" vs "iteration" distinction for SGD. If a dataset has $n=1000$ samples, how many iterations are in one epoch?

4. A practitioner observes that SGD's loss curve bounces around but is trending downward. They switch to BGD, which is smoother but much slower. What is the practical trade-off? For what dataset size is each preferred?

5. Mini-batch GD with batch size $b$: as $b \to 1$ this becomes SGD, and as $b \to n$ this becomes BGD. What happens to (a) gradient variance, (b) memory usage, and (c) wall-clock time per update as $b$ increases?

**Numerical**

6. Implement SGD for logistic regression from scratch. Plot the loss per epoch (averaged over all single-sample updates in that epoch) for 50 epochs. Compare convergence with BGD.

7. Implement mini-batch GD with batch sizes $b \in \{1, 8, 32, 128, n\}$. On a single plot, show the training loss (y-axis) vs. number of samples processed (x-axis) for each batch size.

8. Implement a decaying learning rate $\alpha_t = \alpha_0/(1+t)$. Compare the final training loss after 100 epochs against constant $\alpha$.

**Reflection**

9. SGD is sometimes described as "adding noise" that can help escape local minima. Does this apply to the convex MSE loss for linear regression? Why or why not?